In [3]:
import os
os.environ["SAM3_3D_OBJECTS_ENABLED"] = "true"
os.environ["SPARSE_ATTN_BACKEND"] = "flash_attn"
os.environ["ATTN_BACKEND"] = "flash_attn"

from inference.models.sam3_3d.segment_anything_3d import SegmentAnything3_3D_Objects
from inference.core.entities.requests.sam3_3d import Sam3_3D_Objects_InferenceRequest
from inference.core.entities.responses.sam3_3d import Sam3_3D_Objects_Response
from inference import get_model
import json

def load_coco_masks(annotations_path, image_id):
    with open(annotations_path, 'r') as f:
        coco_data = json.load(f)

    annotations = [
        ann for ann in coco_data['annotations']
        if ann['image_id'] == image_id
    ]

    segmentations = [ann['segmentation'] for ann in annotations]

    return segmentations

image_path = "/home/lee/code/inference/inference/models/sam3_3d/demo-data/train/IMG_6860_jpeg.rf.9fceef9265cef07f0cd4b339527a6756.jpg"
annotations_path = "/home/lee/code/inference/inference/models/sam3_3d/demo-data/train/_annotations.coco.json"
image_id = 0

mask_polygon = load_coco_masks(annotations_path, image_id)

image = {
    "type": "file",
    "value": image_path
}

model = get_model("sam3-3d-objects", api_key="BecfnYVBMM9izzi5ITWp")

request = Sam3_3D_Objects_InferenceRequest(
    image=image,
    mask_input=mask_polygon,
)

print("Running SAM3_3D inference...")
response: Sam3_3D_Objects_Response = model.infer_from_request(request)

print(f"\nSAM3_3D inference completed in {response.time:.2f} seconds!")
print("=" * 80)

# 1. Scene Mesh (GLB)
if response.mesh_glb is not None:
    print(f"\n[Output 1/3] Scene Mesh (GLB format)")
    with open("out_mesh.glb", "wb") as f:
        f.write(response.mesh_glb)
    print(f"  Saved mesh to out_mesh.glb ({len(response.mesh_glb):,} bytes)")
else:
    print(f"\n[Output 1/3] No mesh output")

# 2. Combined Gaussian splatting
if response.gaussian_ply is not None:
    print(f"\n[Output 2/3] Combined Gaussian splatting (PLY format)")
    with open("out_gaussian.ply", "wb") as f:
        f.write(response.gaussian_ply)
    print(f"  Saved gaussian to out_gaussian.ply ({len(response.gaussian_ply):,} bytes)")
else:
    print(f"\n[Output 2/3] No combined gaussian output")

# 3. Individual objects
print(f"\n[Output 3/3] Individual objects ({len(response.objects)} objects)")
objects_metadata = []
for i, obj in enumerate(response.objects):
    print(f"\n  Object {i}:")

    # Save individual mesh
    if obj.mesh_glb is not None:
        filename = f"out_object_{i}_mesh.glb"
        with open(filename, "wb") as f:
            f.write(obj.mesh_glb)
        print(f"    Saved mesh to {filename} ({len(obj.mesh_glb):,} bytes)")

    # Save individual gaussian
    if obj.gaussian_ply is not None:
        filename = f"out_object_{i}_gaussian.ply"
        with open(filename, "wb") as f:
            f.write(obj.gaussian_ply)
        print(f"    Saved gaussian to {filename} ({len(obj.gaussian_ply):,} bytes)")

    # Collect metadata
    obj_metadata = {
        "object_index": i,
        "rotation": obj.metadata.rotation,
        "translation": obj.metadata.translation,
        "scale": obj.metadata.scale,
    }
    objects_metadata.append(obj_metadata)
    print(f"    Metadata: rotation={obj.metadata.rotation is not None}, translation={obj.metadata.translation is not None}, scale={obj.metadata.scale is not None}")

# Save all metadata to file
with open("out_metadata.json", "w") as f:
    json.dump({"objects": objects_metadata}, f, indent=2)
print(f"\n  Saved all metadata to out_metadata.json")

print("\n" + "=" * 80)
print("All outputs saved successfully!")
print("=" * 80)

2026-02-23 16:44:56.077 | WARNING  | tdfy.sam3d_v1.lidra.data.dataset.tdfy.trellis.dataset:__post_init__:80 - No rgb pointmap normalizer provided, using scale + shift 
2026-02-23 16:45:06.347 | WARNING  | tdfy.sam3d_v1.lidra.data.dataset.tdfy.trellis.dataset:__post_init__:80 - No rgb pointmap normalizer provided, using scale + shift 
2026-02-23 16:45:06.392 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:__init__:113 - self.device: cuda:0
2026-02-23 16:45:06.401 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:__init__:114 - CUDA_VISIBLE_DEVICES: 0
2026-02-23 16:45:06.435 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:__init__:117 - Actually using GPU: 0
2026-02-23 16:45:06.469 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:init_pose_decoder:362 - Using pose decoder: ScaleShiftInvariant
2026-02-23 16:45:06.493 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:__init__:162 - Loading model weights...
2026-02-23 16:45:10.044 

Running SAM3_3D inference...


2026-02-23 16:45:48.096 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:374 - InferencePipelinePointMap.run() called
2026-02-23 16:45:48.113 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:merge_image_and_mask:767 - Replacing alpha channel with the provided mask
2026-02-23 16:45:53.171 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:sample_sparse_structure:849 - Sampling sparse structure: inference_steps=25, strength=7, interval=[0, 500], rescale_t=3, cfg_strength_pm=0.0
2026-02-23 16:45:53.260 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:get_condition_input:818 - Running condition embedder ...
2026-02-23 16:45:53.852 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:get_condition_input:822 - Condition embedder finishes!
2026-02-23 16:46:08.071 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:sample_sparse_structure:899 - Downsampled coords from 18626 to 18619
2026-02-23 16:46:08.378 | INFO     |


Loading ..96%

100% done 


2026-02-23 16:46:36.628 | INFO     | tdfy.sam3d_v1.lidra.model.backbone.trellis.utils.postprocessing_utils:to_glb:647 - Baking texture ...
Rendering: 100it [00:01, 63.34it/s]
2026-02-23 16:46:52.660 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:483 - Finished!
2026-02-23 16:46:52.666 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:374 - InferencePipelinePointMap.run() called
2026-02-23 16:46:52.671 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:merge_image_and_mask:767 - Replacing alpha channel with the provided mask
2026-02-23 16:46:55.015 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:sample_sparse_structure:849 - Sampling sparse structure: inference_steps=25, strength=7, interval=[0, 500], rescale_t=3, cfg_strength_pm=0.0
2026-02-23 16:46:55.017 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:get_condition_input:818 - Running condition embedder ...
2026-02-23 16:46:55.290 | INFO     | tdfy

...


Loading ..92%

100% done 


2026-02-23 16:48:20.814 | INFO     | tdfy.sam3d_v1.lidra.model.backbone.trellis.utils.postprocessing_utils:to_glb:647 - Baking texture ...
Rendering: 100it [00:01, 75.35it/s]
2026-02-23 16:48:35.016 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:483 - Finished!
2026-02-23 16:48:35.023 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:374 - InferencePipelinePointMap.run() called
2026-02-23 16:48:35.028 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:merge_image_and_mask:767 - Replacing alpha channel with the provided mask
2026-02-23 16:48:37.345 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:sample_sparse_structure:849 - Sampling sparse structure: inference_steps=25, strength=7, interval=[0, 500], rescale_t=3, cfg_strength_pm=0.0
2026-02-23 16:48:37.348 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline:get_condition_input:818 - Running condition embedder ...
2026-02-23 16:48:37.618 | INFO     | tdfy


Loading ..95%

0% done 


2026-02-23 16:49:06.366 | INFO     | tdfy.sam3d_v1.lidra.model.backbone.trellis.utils.postprocessing_utils:to_glb:647 - Baking texture ...
Rendering: 100it [00:01, 80.14it/s]
2026-02-23 16:49:20.375 | INFO     | tdfy.sam3d_v1.lidra.pipeline.inference_pipeline_pointmap:run:483 - Finished!



SAM3_3D inference completed in 214.10 seconds!

[Output 1/3] Scene Mesh (GLB format)
  Saved mesh to out_mesh.glb (5,815,792 bytes)

[Output 2/3] Combined Gaussian splatting (PLY format)
  Saved gaussian to out_gaussian.ply (94,316,961 bytes)

[Output 3/3] Individual objects (4 objects)

  Object 0:
    Saved mesh to out_object_0_mesh.glb (1,402,548 bytes)
    Saved gaussian to out_object_0_gaussian.ply (40,515,360 bytes)
    Metadata: rotation=True, translation=True, scale=True

  Object 1:
    Saved mesh to out_object_1_mesh.glb (1,310,704 bytes)
    Saved gaussian to out_object_1_gaussian.ply (17,760,928 bytes)
    Metadata: rotation=True, translation=True, scale=True

  Object 2:
    Saved mesh to out_object_2_mesh.glb (1,397,336 bytes)
    Saved gaussian to out_object_2_gaussian.ply (31,743,904 bytes)
    Metadata: rotation=True, translation=True, scale=True

  Object 3:
    Saved mesh to out_object_3_mesh.glb (1,377,392 bytes)
    Saved gaussian to out_object_3_gaussian.ply (4,2